# 07 · Shelf vs offshore within each region
Each sub-region is split at the 200 m isobath (static bathymetry `deptho`). **Benguela shelf** holds the coastal upwelling; **Agulhas shelf** is mostly the Agulhas Bank; **Agulhas offshore** contains the Agulhas Current core just seaward of the shelf break. Input: `scripts/build_zone_series.py`. Surface (0.494 m) reanalysis temperature — model output constrained by observations, not independent observations.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline
from src import modelling
import pandas as pd, numpy as np, xarray as xr, matplotlib.pyplot as plt
visualisation.apply_style()
df = pd.read_csv('../data/processed/thetao_zones_monthly.csv', index_col=0, parse_dates=True).asfreq('MS')
zm = xr.open_dataset('../data/processed/zone_masks.nc')
LABEL = {'benguela_shelf': 'Benguela shelf', 'benguela_offshore': 'Benguela offshore',
         'agulhas_shelf': 'Agulhas shelf', 'agulhas_offshore': 'Agulhas offshore'}
COLORS = {'benguela_shelf': '#1e3a8a', 'benguela_offshore': '#60a5fa',
          'agulhas_shelf': '#f59e0b', 'agulhas_offshore': '#b91c1c'}

## Zone map

In [ ]:
from matplotlib.colors import ListedColormap
zid = xr.zeros_like(zm['deptho']) * np.nan
for i, k in enumerate(LABEL):
    zid = zid.where(zm[k] != 1, i)
fig, ax = plt.subplots(figsize=(6.5, 3.5))
zid.plot(ax=ax, cmap=ListedColormap([COLORS[k] for k in LABEL]), add_colorbar=False, vmin=-0.5, vmax=3.5)
zm['deptho'].plot.contour(ax=ax, levels=[200], colors='k', linewidths=0.6)
ax.axvline(20, color='k', ls='--', lw=0.8)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COLORS[k], label=LABEL[k]) for k in LABEL], fontsize=7, loc='center left', bbox_to_anchor=(1.01, 0.5))
ax.set(title='Zones: 200 m isobath (black) and 20°E split', xlabel='Longitude (°E)', ylabel='Latitude (°N)')
ax.grid(False); fig.tight_layout(); visualisation.save(fig, '07_zone_map')

## Climatology and seasonal cycle

In [ ]:
stats = pd.concat({k: preprocessing.describe_series(df[k])['value'] for k in df}, axis=1)
stats.to_csv('../outputs/tables/07_zone_stats.csv')
stats

In [ ]:
clims = pd.DataFrame({k: preprocessing.monthly_climatology(df[k]) for k in df})
fig, ax = plt.subplots(figsize=(6, 3.2))
for k in clims:
    ax.plot(clims.index, clims[k], marker='o', ms=3, color=COLORS[k], label=LABEL[k])
ax.set(title='Monthly climatology 1993–2025', xlabel='Month', ylabel='Temperature (°C)', xticks=range(1, 13))
ax.legend(fontsize=7, ncol=2); visualisation.source_note(ax); fig.tight_layout()
visualisation.save(fig, '07_zone_seasonality')

## Trends (annual, all months)
HAC (Newey–West) 95 % confidence intervals.

In [ ]:
anoms = pd.DataFrame({k: preprocessing.anomalies(df[k]) for k in df})
trends = pd.DataFrame({k: preprocessing.hac_trend(anoms[k]) for k in anoms}).T
trends.to_csv('../outputs/tables/07_zone_trends.csv')
trends.round(4)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8, 7.5), sharex=True)
for ax, k in zip(axes, anoms):
    a = anoms[k]; ax.fill_between(a.index, 0, a, where=a > 0, color='#c53030', lw=0)
    ax.fill_between(a.index, 0, a, where=a < 0, color='#2b6cb0', lw=0)
    ax.plot(a.rolling(12, center=True).mean(), color='k', lw=1)
    t = trends.loc[k]
    ax.set_title(f"{LABEL[k]}: {t.trend_degC_per_decade:+.2f} ± {t.ci95:.2f} °C/decade", fontsize=9)
    ax.set_ylabel('°C')
visualisation.source_note(axes[-1]); fig.tight_layout()
visualisation.save(fig, '07_zone_anomalies')

## Seasonal trends
The southern Benguela upwelling peaks in austral spring–summer (SON–DJF). Trends from yearly seasonal means; DJF is labelled by its December.

In [ ]:
st = pd.concat({k: preprocessing.seasonal_trends(anoms[k]) for k in anoms}, names=['zone', 'season'])
st.to_csv('../outputs/tables/07_zone_seasonal_trends.csv')
tbl = st['trend_degC_per_decade'].unstack().round(3)[list(preprocessing.SEASONS)]
sig = (st['p_value'] < 0.05).unstack()[list(preprocessing.SEASONS)]
tbl.astype(str) + sig.replace({True: ' *', False: ''})

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.arange(4); w = 0.2
for i, k in enumerate(anoms):
    d = st.loc[k].loc[list(preprocessing.SEASONS)]
    ax.bar(x + (i - 1.5) * w, d['trend_degC_per_decade'], w, yerr=d['ci95'], color=COLORS[k],
           label=LABEL[k], capsize=2, error_kw={'lw': 0.7})
ax.axhline(0, color='0.3', lw=0.8)
ax.set(xticks=x, xticklabels=list(preprocessing.SEASONS), ylabel='°C per decade',
       title='Seasonal SST trends 1993–2025 (95 % CI)')
ax.legend(fontsize=7, ncol=2); visualisation.source_note(ax); fig.tight_layout()
visualisation.save(fig, '07_zone_seasonal_trends')

## Cross-shelf temperature difference (upwelling proxy)
ΔT = offshore − shelf. Stronger upwelling cools the shelf relative to offshore, raising ΔT. This is a thermal proxy only; wind stress and vertical velocity would be needed to attribute changes.

In [ ]:
dT = pd.DataFrame({r: df[f'{r}_offshore'] - df[f'{r}_shelf'] for r in ('benguela', 'agulhas')})
dT_trend = pd.DataFrame({r: preprocessing.hac_trend(preprocessing.anomalies(dT[r])) for r in dT}).T
dT_seas = pd.concat({r: preprocessing.seasonal_trends(preprocessing.anomalies(dT[r])) for r in dT})
dT_trend.to_csv('../outputs/tables/07_crossshelf_dT_trends.csv')
dT_seas.to_csv('../outputs/tables/07_crossshelf_dT_seasonal_trends.csv')
display(dT.groupby(dT.index.month).mean().round(2).T)
display(dT_trend.round(4)); dT_seas.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for r, c in (('benguela', '#1e3a8a'), ('agulhas', '#b91c1c')):
    ax.plot(dT[r], color=c, lw=0.6, alpha=0.4)
    ax.plot(dT[r].rolling(12, center=True).mean(), color=c, lw=1.5, label=f'{r.title()} (12-month mean)')
ax.set(title='Cross-shelf ΔT (offshore − shelf)', ylabel='ΔT (°C)')
ax.legend(fontsize=8); visualisation.source_note(ax); fig.tight_layout()
visualisation.save(fig, '07_crossshelf_dT')

## One-month-ahead skill by zone (validation 2016–2019; test untouched)

In [ ]:
m = settings['modelling']; rows = {}
for k in df:
    sp = modelling.chronological_split(modelling.make_features(df[k], m['lags']),
                                       m['train_end'], m['validation_end'], m['test_end'])
    rows[k] = modelling.evaluate(sp, settings['project']['random_seed'], on='validation')
skill = pd.concat(rows, names=['zone', 'model'])
skill.round(3).to_csv('../outputs/tables/07_zone_validation_metrics.csv')
skill['RMSE'].unstack().round(3)